# Lecture 8: Building a Backtesting Engine

In this lecture, we'll build the infrastructure for backtesting a trading strategy. This will serve as the foundation for **Homework 4**, where you'll implement your own prediction model and allocation strategy.

## What We'll Cover

1. **Understanding the Trading Rules** - How positions work in our backtest
2. **The Backtester Architecture** - Classes and data structures
3. **Position Management** - Tracking entries, exits, and P&L
4. **The Main Loop** - How the backtest runs day by day
5. **Benchmark Calculation** - S&P 500 Total Return comparison
6. **Your Tasks** - What you need to implement for HW4

---

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Protocol, Any
import warnings
warnings.filterwarnings('ignore')

---

## Part 1: Loading and Exploring the Data

First, let's load our `signals` table and understand its structure. Each row represents a **ticker + date** combination with the stock's adjusted close price and various fundamental features.

The homework database is distributed separately through bCourses. If `data/data.db` is absent, we use the same small synthetic dataset as Part 9 so the lecture can run end to end. This sample is **only for learning the backtester**, not for training or evaluating HW4.

In [ ]:
def create_demo_signals():
    """Create a small dummy signals table for testing the backtester.
    
    Returns:
        DataFrame with 100 trading days, 5 tickers, and dummy features.
    """
    np.random.seed(123)  # For reproducibility
    
    # 5 tickers including benchmark
    tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', '^SP500TR']
    
    # 100 trading days (roughly 5 months, skipping weekends)
    # This is long enough for positions to close (90 calendar day holding period)
    start_date = pd.Timestamp('2016-04-01')
    all_days = pd.date_range(start=start_date, periods=150, freq='D')
    # Filter to weekdays only (simple approximation of trading days)
    trading_days = [d for d in all_days if d.weekday() < 5][:100]
    
    rows = []
    
    # Generate price series for each ticker
    for ticker in tickers:
        # Starting price
        if ticker == '^SP500TR':
            price = 1000.0  # Index level
        else:
            price = np.random.uniform(50, 200)
        
        for date in trading_days:
            # Random daily return between -3% and +3%
            daily_return = np.random.uniform(-0.03, 0.03)
            price = price * (1 + daily_return)
            
            # Two dummy feature columns
            feature_1 = np.random.uniform(-1, 1)  # e.g., momentum score
            feature_2 = np.random.uniform(0, 100)  # e.g., some fundamental metric
            
            rows.append({
                'symbol': ticker,
                'date': date,
                'adj_close': round(price, 2),
                'momentum_score': round(feature_1, 4),
                'pe_ratio': round(feature_2, 2)
            })
    
    return pd.DataFrame(rows)


# The real homework database is distributed separately through bCourses.
from pathlib import Path

db_path = Path('data/data.db')
if db_path.is_file():
    with sqlite3.connect(f'{db_path.resolve().as_uri()}?mode=ro', uri=True) as conn:
        signals = pd.read_sql_query('SELECT * FROM signals', conn)
    print('Loaded the course signals dataset.')
else:
    signals = create_demo_signals()
    print('DEMO DATA ONLY: the course signals database is not installed.')
    print('For HW4, place the real database at Lectures/Lecture 8/data/data.db.')

print(f'Shape: {signals.shape}')
print(f'Columns: {signals.columns.tolist()[:20]}')
print(f'Total columns: {len(signals.columns)}')


In [ ]:
# Check the date column type and convert if needed
signals['date'] = pd.to_datetime(signals['date'])

print(f"Date range: {signals['date'].min()} to {signals['date'].max()}")
print(f"Number of unique symbols: {signals['symbol'].nunique()}")
print(f"Number of unique dates: {signals['date'].nunique()}")

In [ ]:
# Look at a sample of the data
signals.head()

### Key Column: `adj_close`

The `adj_close` column is the **adjusted closing price** - this is what we use for:
- Entering positions (buy at close)
- Exiting positions (sell at close)
- Calculating returns

All other columns are **features** you can use for your prediction model.

In [ ]:
# Verify adj_close exists and check for issues
print(f"adj_close missing values: {signals['adj_close'].isna().sum()}")
print(f"adj_close stats:\n{signals['adj_close'].describe()}")

---

## Part 2: Understanding the Trading Rules

Before we build the backtester, let's be crystal clear on the rules:

| Rule | Description |
|------|-------------|
| **Entry** | Buy at the close price on any trading day |
| **Holding Period** | Exactly **90 calendar days** |
| **Exit** | Sell at close price on day 90 (or next trading day if market closed) |
| **Cash** | Consumed on entry, returned on exit |

### Example Timeline

```
April 1, 2020:  Buy AAPL at $63.57 (invest $10,000 → 157.3 shares)
                Scheduled exit: June 30, 2020 (90 calendar days later)
                
June 30, 2020:  Sell AAPL at $91.20
                Return = ($91.20 - $63.57) / $63.57 = 43.5%
                Cash received: 157.3 × $91.20 = $14,346
```

---

## Part 3: The Position Class

We need a data structure to track each position. A **position** represents a single investment in one stock.

We'll use Python's `dataclass` for clean, readable code:

In [ ]:
@dataclass
class Position:
    """Represents a single stock position.
    
    Attributes:
        ticker: Stock symbol (e.g., 'AAPL')
        entry_date: Date the position was opened
        entry_price: Price paid per share
        shares: Number of shares (can be fractional)
        scheduled_exit_date: 90 calendar days after entry
    """
    ticker: str
    entry_date: pd.Timestamp
    entry_price: float
    shares: float
    scheduled_exit_date: pd.Timestamp = field(init=False)
    
    def __post_init__(self):
        """Calculate scheduled exit date (90 calendar days after entry)."""
        self.scheduled_exit_date = self.entry_date + timedelta(days=90)
    
    @property
    def cost_basis(self) -> float:
        """Total amount invested in this position."""
        return self.entry_price * self.shares
    
    def market_value(self, current_price: float) -> float:
        """Current market value of the position."""
        return current_price * self.shares
    
    def pnl(self, current_price: float) -> float:
        """Profit/loss at current price."""
        return self.market_value(current_price) - self.cost_basis
    
    def return_pct(self, current_price: float) -> float:
        """Percentage return at current price."""
        return (current_price - self.entry_price) / self.entry_price

In [ ]:
# Example: Create a position
example_position = Position(
    ticker='AAPL',
    entry_date=pd.Timestamp('2020-04-01'),
    entry_price=63.57,
    shares=157.3
)

print(f"Position: {example_position.ticker}")
print(f"Entry: {example_position.entry_date.date()} @ ${example_position.entry_price:.2f}")
print(f"Shares: {example_position.shares:.2f}")
print(f"Cost basis: ${example_position.cost_basis:,.2f}")
print(f"Scheduled exit: {example_position.scheduled_exit_date.date()}")
print(f"\nIf current price is $91.20:")
print(f"  Market value: ${example_position.market_value(91.20):,.2f}")
print(f"  P&L: ${example_position.pnl(91.20):,.2f}")
print(f"  Return: {example_position.return_pct(91.20):.1%}")

---

## Part 4: The Model Protocol

Your prediction model needs to follow a specific interface so it can plug into the backtester. We'll define a **Protocol** (like an interface in other languages) that specifies what methods your model must have.

This is what **you will implement** for HW4:

In [ ]:
class TradingModel(Protocol):
    """Protocol defining the interface for trading models.
    
    Your model must implement these methods to work with the backtester.
    """
    
    def fit(self, X: pd.DataFrame, y: pd.Series) -> None:
        """Train the model on historical data.
        
        Args:
            X: Feature DataFrame (rows = ticker-dates, columns = features)
            y: Target Series (90-day forward returns)
        """
        ...
    
    def predict(self, X: pd.DataFrame) -> pd.Series:
        """Predict 90-day returns for given features.
        
        Args:
            X: Feature DataFrame for current day (rows = tickers)
            
        Returns:
            Series with predicted 90-day return for each ticker.
            Index should be ticker symbols.
        """
        ...

---

## Part 5: The Backtester Class

Now for the main event - the `Backtester` class. This orchestrates everything:

1. **Tracks cash** - How much money is available to invest
2. **Manages positions** - Opens and closes positions
3. **Runs day-by-day** - Iterates through the test period
4. **Calls your model** - Gets predictions each day
5. **Calls your allocator** - Decides how much to invest
6. **Records history** - Stores data for analysis

In [ ]:
class Backtester:
    """A backtesting engine for 90-day holding period strategies.
    
    This class handles:
    - Position tracking and automatic exit after 90 days
    - Cash management
    - Daily iteration through the test period
    - Model prediction calls
    - Benchmark calculation
    
    You (the student) need to provide:
    - A trained model with .predict() method
    - An allocation function to decide position sizes
    """
    
    def __init__(
        self,
        signals: pd.DataFrame,
        initial_cash: float = 1_000_000,
        benchmark_ticker: str = '^SP500TR'
    ):
        """Initialize the backtester.
        
        Args:
            signals: DataFrame with columns ['symbol', 'date', 'adj_close', ...features...]
            initial_cash: Starting capital (default $1,000,000)
            benchmark_ticker: Ticker for benchmark comparison (default S&P 500 Total Return)
        """
        self.signals = signals.copy()
        self.signals['date'] = pd.to_datetime(self.signals['date'])
        self.initial_cash = initial_cash
        self.benchmark_ticker = benchmark_ticker
        
        # State variables (reset on each run)
        self.cash: float = initial_cash
        self.positions: List[Position] = []
        self.closed_positions: List[Dict] = []
        self.history: List[Dict] = []  # Daily snapshots
        
        # Build lookup structures for fast access
        self._build_price_lookup()
        self._build_trading_days()
    
    def _build_price_lookup(self):
        """Create a fast lookup for prices: (symbol, date) -> adj_close."""
        self.price_lookup = (
            self.signals
            .set_index(['symbol', 'date'])['adj_close']
            .to_dict()
        )
        
        # Also store last known price for each symbol (for delisted stocks)
        self.last_price = (
            self.signals
            .sort_values('date')
            .groupby('symbol')['adj_close']
            .last()
            .to_dict()
        )
        
        # Store last available date for each symbol
        self.last_date = (
            self.signals
            .groupby('symbol')['date']
            .max()
            .to_dict()
        )
    
    def _build_trading_days(self):
        """Build sorted list of all trading days."""
        self.trading_days = sorted(self.signals['date'].unique())
        self.trading_days_set = set(self.trading_days)
    
    def get_price(self, ticker: str, date: pd.Timestamp) -> Optional[float]:
        """Get the price for a ticker on a specific date.
        
        If the exact date isn't available (weekend, holiday, or delisted),
        returns the last known price.
        
        Args:
            ticker: Stock symbol
            date: Date to look up
            
        Returns:
            Price if found, None if ticker never existed in data.
        """
        # Try exact match first
        price = self.price_lookup.get((ticker, date))
        if price is not None:
            return price
        
        # If stock is delisted (date > last available date), use last price
        if ticker in self.last_date:
            if date > self.last_date[ticker]:
                return self.last_price.get(ticker)
        
        # For weekends/holidays, find the next trading day
        next_trading_day = self._find_next_trading_day(date)
        if next_trading_day:
            return self.price_lookup.get((ticker, next_trading_day))
        
        return None
    
    def _find_next_trading_day(self, date: pd.Timestamp) -> Optional[pd.Timestamp]:
        """Find the next trading day on or after the given date."""
        for td in self.trading_days:
            if td >= date:
                return td
        return None
    
    def _find_actual_exit_date(self, scheduled_date: pd.Timestamp) -> pd.Timestamp:
        """Find the actual exit date (next trading day on or after scheduled)."""
        next_td = self._find_next_trading_day(scheduled_date)
        return next_td if next_td else scheduled_date
    
    def get_data_for_date(self, date: pd.Timestamp) -> pd.DataFrame:
        """Get all stock data for a specific date.
        
        Args:
            date: The date to get data for
            
        Returns:
            DataFrame with one row per symbol, indexed by symbol.
        """
        day_data = self.signals[self.signals['date'] == date].copy()
        if len(day_data) > 0:
            day_data = day_data.set_index('symbol')
        return day_data
    
    def get_training_data(
        self,
        start_date: str = '2010-01-01',
        end_date: str = '2015-12-31'
    ) -> pd.DataFrame:
        """Get data for the training period.
        
        Args:
            start_date: Training period start (default '2010-01-01')
            end_date: Training period end (default '2015-12-31')
            
        Returns:
            DataFrame filtered to the training period.
        """
        start = pd.to_datetime(start_date)
        end = pd.to_datetime(end_date)
        mask = (self.signals['date'] >= start) & (self.signals['date'] <= end)
        return self.signals[mask].copy()
    
    def get_historical_data_up_to(
        self,
        date: pd.Timestamp,
        lookback_days: Optional[int] = None
    ) -> pd.DataFrame:
        """Get all historical data up to (but not including) a date.
        
        Useful for retraining models during the test period.
        
        Args:
            date: Get data before this date
            lookback_days: Optional limit on how far back to look
            
        Returns:
            DataFrame with historical data.
        """
        mask = self.signals['date'] < date
        if lookback_days:
            start_date = date - timedelta(days=lookback_days)
            mask = mask & (self.signals['date'] >= start_date)
        return self.signals[mask].copy()
    
    def open_position(self, ticker: str, date: pd.Timestamp, dollars: float) -> Optional[Position]:
        """Open a new position.
        
        Args:
            ticker: Stock to buy
            date: Entry date
            dollars: Amount to invest
            
        Returns:
            The Position object if successful, None if failed.
        """
        # Validate we have enough cash
        if dollars > self.cash:
            return None
        
        if dollars <= 0:
            return None
        
        # Get the entry price
        price = self.get_price(ticker, date)
        if price is None or price <= 0:
            return None
        
        # Calculate shares (fractional allowed)
        shares = dollars / price
        
        # Create position and deduct cash
        position = Position(
            ticker=ticker,
            entry_date=date,
            entry_price=price,
            shares=shares
        )
        
        self.positions.append(position)
        self.cash -= dollars
        
        return position
    
    def close_position(self, position: Position, date: pd.Timestamp) -> Dict:
        """Close a position and return cash.
        
        Args:
            position: The position to close
            date: Exit date
            
        Returns:
            Dict with position details and P&L.
        """
        # Get exit price (handles delisted stocks)
        exit_price = self.get_price(position.ticker, date)
        
        # If still no price, use entry price (no gain/loss)
        if exit_price is None:
            exit_price = position.entry_price
        
        # Calculate proceeds and return cash
        proceeds = position.shares * exit_price
        self.cash += proceeds
        
        # Remove from active positions
        if position in self.positions:
            self.positions.remove(position)
        
        # Record closed position
        closed_record = {
            'ticker': position.ticker,
            'entry_date': position.entry_date,
            'exit_date': date,
            'entry_price': position.entry_price,
            'exit_price': exit_price,
            'shares': position.shares,
            'cost_basis': position.cost_basis,
            'proceeds': proceeds,
            'pnl': proceeds - position.cost_basis,
            'return_pct': (exit_price - position.entry_price) / position.entry_price
        }
        
        self.closed_positions.append(closed_record)
        
        return closed_record
    
    def _process_exits(self, current_date: pd.Timestamp) -> List[Dict]:
        """Process any positions that need to be closed today.
        
        A position is closed if:
        - Current date >= scheduled exit date, OR
        - Stock is delisted (no data after last available date)
        
        Args:
            current_date: Today's date in the simulation
            
        Returns:
            List of closed position records.
        """
        closed_today = []
        positions_to_close = []
        
        for position in self.positions:
            # Check if scheduled to exit
            actual_exit_date = self._find_actual_exit_date(position.scheduled_exit_date)
            if current_date >= actual_exit_date:
                positions_to_close.append((position, current_date))
                continue
            
            # Check if delisted
            last_available = self.last_date.get(position.ticker)
            if last_available and current_date > last_available:
                # Stock delisted - close at last available price
                positions_to_close.append((position, last_available))
        
        # Close positions
        for position, exit_date in positions_to_close:
            closed_record = self.close_position(position, exit_date)
            closed_today.append(closed_record)
        
        return closed_today
    
    def calculate_portfolio_value(self, date: pd.Timestamp) -> float:
        """Calculate total portfolio value (cash + positions marked to market).
        
        Args:
            date: Date to calculate value for
            
        Returns:
            Total portfolio value in dollars.
        """
        positions_value = 0.0
        for position in self.positions:
            price = self.get_price(position.ticker, date)
            if price:
                positions_value += position.market_value(price)
            else:
                # If no price available, use cost basis
                positions_value += position.cost_basis
        
        return self.cash + positions_value
    
    def _record_daily_snapshot(self, date: pd.Timestamp, closed_today: List, opened_today: List):
        """Record daily state for later analysis."""
        snapshot = {
            'date': date,
            'cash': self.cash,
            'num_positions': len(self.positions),
            'positions_value': sum(
                p.market_value(self.get_price(p.ticker, date) or p.entry_price)
                for p in self.positions
            ),
            'portfolio_value': self.calculate_portfolio_value(date),
            'positions_closed': len(closed_today),
            'positions_opened': len(opened_today),
        }
        self.history.append(snapshot)
    
    def calculate_benchmark_return(self, start_date: str, end_date: str) -> Dict:
        """Calculate buy-and-hold benchmark return for S&P 500.
        
        Args:
            start_date: Start of investment period
            end_date: End of investment period
            
        Returns:
            Dict with benchmark statistics.
        """
        start = pd.to_datetime(start_date)
        end = pd.to_datetime(end_date)
        
        # Get benchmark prices
        benchmark_data = self.signals[
            (self.signals['symbol'] == self.benchmark_ticker) &
            (self.signals['date'] >= start) &
            (self.signals['date'] <= end)
        ].sort_values('date')
        
        if len(benchmark_data) == 0:
            return {'error': f'No data found for benchmark ticker {self.benchmark_ticker}'}
        
        start_price = benchmark_data.iloc[0]['adj_close']
        end_price = benchmark_data.iloc[-1]['adj_close']
        
        # Calculate metrics
        total_return = (end_price - start_price) / start_price
        years = (benchmark_data.iloc[-1]['date'] - benchmark_data.iloc[0]['date']).days / 365.25
        annualized_return = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0
        
        # Final value if invested $1M at start
        final_value = self.initial_cash * (1 + total_return)
        
        # Get daily values for the benchmark
        shares = self.initial_cash / start_price
        benchmark_data = benchmark_data.copy()
        benchmark_data['benchmark_value'] = shares * benchmark_data['adj_close']
        
        return {
            'start_date': benchmark_data.iloc[0]['date'],
            'end_date': benchmark_data.iloc[-1]['date'],
            'start_price': start_price,
            'end_price': end_price,
            'total_return': total_return,
            'annualized_return': annualized_return,
            'final_value': final_value,
            'daily_values': benchmark_data[['date', 'benchmark_value']].copy()
        }
    
    def run(
        self,
        model: Any,
        allocate_fn,
        start_date: str = '2016-04-01',
        end_date: str = '2025-12-31',
        retrain_fn = None,
        retrain_frequency: Optional[int] = None,
        feature_columns: Optional[List[str]] = None,
        verbose: bool = True
    ) -> Dict:
        """Run the backtest.
        
        Args:
            model: Your trained model with .predict(X) method
            allocate_fn: Function(predictions, available_cash, day_data) -> Dict[ticker, dollars]
            start_date: Test period start (default '2016-04-01')
            end_date: Test period end (default '2025-12-31')
            retrain_fn: Optional function(model, historical_data) to retrain model
            retrain_frequency: How often to retrain (in trading days). None = never retrain.
            feature_columns: List of column names to use as features for prediction.
                           If None, uses all columns except ['symbol', 'date', 'adj_close'].
            verbose: Whether to print progress
            
        Returns:
            Dict with backtest results and history.
        """
        # Reset state
        self.cash = self.initial_cash
        self.positions = []
        self.closed_positions = []
        self.history = []
        
        # Parse dates
        start = pd.to_datetime(start_date)
        end = pd.to_datetime(end_date)
        
        # Get trading days in test period
        test_days = [d for d in self.trading_days if start <= d <= end]
        
        if verbose:
            print(f"Running backtest from {start_date} to {end_date}")
            print(f"Trading days: {len(test_days)}")
            print(f"Initial cash: ${self.initial_cash:,.2f}")
            print("-" * 50)
        
        # Determine feature columns
        if feature_columns is None:
            exclude_cols = ['symbol', 'date', 'adj_close']
            feature_columns = [c for c in self.signals.columns if c not in exclude_cols]
        
        # Main loop
        for i, current_date in enumerate(test_days):
            # 1. Process exits first (positions held for 90 days)
            closed_today = self._process_exits(current_date)
            
            # 2. Optional: Retrain model
            if retrain_fn and retrain_frequency and i > 0 and i % retrain_frequency == 0:
                historical_data = self.get_historical_data_up_to(current_date)
                retrain_fn(model, historical_data)
                if verbose:
                    print(f"[{current_date.date()}] Model retrained on {len(historical_data)} rows")
            
            # 3. Get today's data and make predictions
            day_data = self.get_data_for_date(current_date)
            
            opened_today = []
            if len(day_data) > 0 and self.cash > 0:
                # Prepare features for prediction
                available_features = [c for c in feature_columns if c in day_data.columns]
                X_today = day_data[available_features]
                
                # Get predictions from model
                try:
                    predictions = model.predict(X_today)
                    # Ensure predictions is a Series with symbol index
                    if not isinstance(predictions, pd.Series):
                        predictions = pd.Series(predictions, index=day_data.index)
                except Exception as e:
                    if verbose:
                        print(f"[{current_date.date()}] Prediction error: {e}")
                    predictions = pd.Series(dtype=float)
                
                # 4. Get allocation decisions from student's function
                if len(predictions) > 0:
                    allocations = allocate_fn(predictions, self.cash, day_data)
                    
                    # 5. Open new positions
                    for ticker, dollars in allocations.items():
                        if dollars > 0 and self.cash >= dollars:
                            position = self.open_position(ticker, current_date, dollars)
                            if position:
                                opened_today.append(position)
            
            # 6. Record daily snapshot
            self._record_daily_snapshot(current_date, closed_today, opened_today)
            
            # Progress update
            if verbose and (i + 1) % 500 == 0:
                pv = self.calculate_portfolio_value(current_date)
                print(f"[{current_date.date()}] Day {i+1}/{len(test_days)} | "
                      f"Portfolio: ${pv:,.0f} | Positions: {len(self.positions)} | "
                      f"Cash: ${self.cash:,.0f}")
        
        # Final summary
        final_date = test_days[-1] if test_days else end
        final_value = self.calculate_portfolio_value(final_date)
        
        # Calculate benchmark
        benchmark = self.calculate_benchmark_return(start_date, end_date)
        
        if verbose:
            print("-" * 50)
            print(f"Backtest complete!")
            print(f"Final portfolio value: ${final_value:,.2f}")
            print(f"Total positions opened: {len(self.closed_positions) + len(self.positions)}")
            print(f"Positions still open: {len(self.positions)}")
        
        # Build results
        results = {
            'initial_cash': self.initial_cash,
            'final_value': final_value,
            'final_cash': self.cash,
            'open_positions': self.positions.copy(),
            'closed_positions': pd.DataFrame(self.closed_positions),
            'history': pd.DataFrame(self.history),
            'benchmark': benchmark
        }
        
        return results

---

## Part 6: Understanding the Backtest Flow

Let's walk through what happens on each day of the backtest:

```
For each trading day:
    │
    ├─► 1. PROCESS EXITS
    │      - Check each open position
    │      - If 90 days have passed → close position, add cash
    │      - If stock delisted → close at last price
    │
    ├─► 2. RETRAIN MODEL (optional)
    │      - If retrain_frequency is set and it's time
    │      - Call retrain_fn(model, historical_data)
    │
    ├─► 3. GET PREDICTIONS
    │      - Get today's data for all stocks
    │      - Call model.predict(features) → predicted returns
    │
    ├─► 4. ALLOCATE CASH (YOUR FUNCTION)
    │      - Call allocate_fn(predictions, available_cash, day_data)
    │      - Returns dict: {ticker: dollars_to_invest}
    │
    ├─► 5. OPEN POSITIONS
    │      - For each allocation, buy shares at close price
    │      - Deduct cash
    │
    └─► 6. RECORD HISTORY
           - Save daily snapshot for analysis
```

---

## Part 7: Your Tasks for HW4 (Stubs)

Here's what you need to implement. The cells below are **stubs** - replace the `pass` statements with your code.

### Task 1: Build Your Prediction Model

Create a model that:
1. Takes features as input
2. Predicts 90-day forward returns
3. Has `.fit(X, y)` and `.predict(X)` methods (sklearn-compatible)

In [ ]:
# =============================================================================
# TASK 1: YOUR PREDICTION MODEL
# =============================================================================
# 
# Create a model class or use sklearn Pipeline.
# 
# Requirements:
#   - fit(X, y): Train on features X and target y (90-day returns)
#   - predict(X): Return predicted 90-day returns for each row
#
# Example using sklearn:
#   from sklearn.pipeline import Pipeline
#   from sklearn.preprocessing import StandardScaler
#   from sklearn.linear_model import Ridge
#   
#   model = Pipeline([
#       ('scaler', StandardScaler()),
#       ('regressor', Ridge(alpha=1.0))
#   ])
#
# =============================================================================

class YourModel:
    """Your prediction model for 90-day stock returns.
    
    TODO: Implement this class or use an sklearn Pipeline instead.
    """
    
    def __init__(self):
        # TODO: Initialize your model
        # Example: self.model = RandomForestRegressor(n_estimators=100)
        pass
    
    def fit(self, X: pd.DataFrame, y: pd.Series) -> 'YourModel':
        """Train the model.
        
        Args:
            X: Feature DataFrame (each row is a ticker-date)
            y: Target Series (90-day forward returns)
            
        Returns:
            self (for method chaining)
        """
        # TODO: Implement training logic
        # Example: self.model.fit(X.fillna(0), y)
        pass
        return self
    
    def predict(self, X: pd.DataFrame) -> pd.Series:
        """Predict 90-day returns.
        
        Args:
            X: Feature DataFrame for prediction
            
        Returns:
            Series of predicted returns, indexed by ticker.
        """
        # TODO: Implement prediction logic
        # Example: 
        #   predictions = self.model.predict(X.fillna(0))
        #   return pd.Series(predictions, index=X.index)
        pass

### Task 2: Prepare Training Data and Train Your Model

You need to:
1. Calculate the **target variable** (90-day forward return)
2. Select and engineer **features**
3. Train your model on 2010-2015 data

In [ ]:
# =============================================================================
# TASK 2: PREPARE TRAINING DATA
# =============================================================================
#
# Calculate the target variable: 90-day forward return
# target = (price_t+90 - price_t) / price_t
#
# CRITICAL: Do not use any future data when creating features!
#
# =============================================================================

def prepare_training_data(signals: pd.DataFrame) -> tuple:
    """Prepare features (X) and target (y) for model training.
    
    Args:
        signals: The full signals DataFrame
        
    Returns:
        Tuple of (X_train, y_train) for the training period (2010-2015)
    """
    # TODO: Implement this function
    # 
    # Steps:
    # 1. Calculate 90-day forward return as target
    #    - Group by symbol, shift adj_close by -90 days
    #    - target = (future_price - current_price) / current_price
    #
    # 2. Select features (all columns except symbol, date, adj_close, target)
    #
    # 3. Filter to training period (2010-01-01 to 2015-12-31)
    #
    # 4. Drop rows with missing target (last 90 days of each stock)
    #
    # 5. Return X_train (features) and y_train (target)
    #
    pass

# Example usage (uncomment when implemented):
# X_train, y_train = prepare_training_data(signals)
# print(f"Training samples: {len(X_train)}")
# print(f"Features: {X_train.columns.tolist()[:10]}...")

In [ ]:
# =============================================================================
# Train your model
# =============================================================================

# TODO: Uncomment and run after implementing prepare_training_data and YourModel
#
# X_train, y_train = prepare_training_data(signals)
# model = YourModel()
# model.fit(X_train, y_train)
# print("Model trained!")

### Task 3: Implement Your Allocation Function

This function decides:
- **Which stocks** to buy based on predictions
- **How much** to invest in each stock

The backtester calls this function every trading day.

In [ ]:
# =============================================================================
# TASK 3: YOUR ALLOCATION FUNCTION
# =============================================================================
#
# This function is called every trading day with:
#   - predictions: Series of predicted 90-day returns (index = ticker)
#   - available_cash: How much cash you have to invest
#   - day_data: Full DataFrame of today's data (for additional info)
#
# Return: Dict mapping ticker -> dollars to invest
#         Example: {'AAPL': 50000, 'MSFT': 30000}
#
# Strategy ideas:
#   - Invest in top N stocks by predicted return
#   - Equal-weight vs. prediction-weighted allocation
#   - Minimum prediction threshold to invest
#   - Diversification constraints
#
# =============================================================================

def allocate_cash(
    predictions: pd.Series,
    available_cash: float,
    day_data: pd.DataFrame
) -> Dict[str, float]:
    """Decide how to allocate available cash based on predictions.
    
    Args:
        predictions: Predicted 90-day returns for each stock (index = ticker)
        available_cash: Cash available to invest
        day_data: Today's full data (includes adj_close and all features)
        
    Returns:
        Dict mapping ticker symbol to dollar amount to invest.
        Tickers not in the dict (or with 0 value) will not be bought.
    """
    # TODO: Implement your allocation strategy
    #
    # Example strategy (buy top 5 predictions, equal weight):
    #
    #   # Only consider positive predictions
    #   positive_preds = predictions[predictions > 0].sort_values(ascending=False)
    #   
    #   # Take top 5
    #   top_stocks = positive_preds.head(5).index.tolist()
    #   
    #   # Equal weight
    #   if len(top_stocks) > 0:
    #       dollars_each = available_cash / len(top_stocks)
    #       return {ticker: dollars_each for ticker in top_stocks}
    #   
    #   return {}
    #
    pass
    return {}  # Remove this line when you implement

### Task 4 (Optional): Implement Model Retraining

If you want to retrain your model periodically during the test period:

In [ ]:
# =============================================================================
# TASK 4 (OPTIONAL): MODEL RETRAINING FUNCTION
# =============================================================================
#
# If you want to retrain your model periodically, implement this function.
# It will be called every `retrain_frequency` trading days.
#
# The historical_data includes ALL data up to (but not including) current day.
#
# =============================================================================

def retrain_model(model: Any, historical_data: pd.DataFrame) -> None:
    """Retrain the model on updated historical data.
    
    Args:
        model: Your model instance
        historical_data: All data from start through yesterday
        
    Note:
        This function should modify the model in-place.
        You need to recalculate targets and features from historical_data.
    """
    # TODO: Implement retraining logic
    #
    # Example:
    #   X, y = prepare_features_and_target(historical_data)
    #   model.fit(X, y)
    #
    pass

### Task 5: Performance Metrics and Visualization

After running the backtest, you need to calculate:
1. Final portfolio value
2. Annualized return
3. Maximum drawdown
4. Sharpe ratio

And create a cumulative return plot comparing your strategy to the benchmark.

In [ ]:
# =============================================================================
# TASK 5: PERFORMANCE METRICS
# =============================================================================
#
# After running the backtest, calculate these metrics from the results.
#
# The backtest returns a dict with:
#   - 'history': DataFrame with daily snapshots (date, portfolio_value, cash, etc.)
#   - 'benchmark': Dict with benchmark info including 'daily_values' DataFrame
#   - 'final_value': Final portfolio value
#   - 'initial_cash': Starting capital
#
# =============================================================================

def calculate_annualized_return(start_value: float, end_value: float, years: float) -> float:
    """Calculate annualized return.
    
    Args:
        start_value: Initial portfolio value
        end_value: Final portfolio value
        years: Number of years
        
    Returns:
        Annualized return as a decimal (e.g., 0.10 for 10%)
    """
    # TODO: Implement
    # Formula: (end/start)^(1/years) - 1
    pass


def calculate_max_drawdown(portfolio_values: pd.Series) -> float:
    """Calculate maximum drawdown.
    
    Args:
        portfolio_values: Series of daily portfolio values
        
    Returns:
        Maximum drawdown as a negative decimal (e.g., -0.20 for 20% drawdown)
    """
    # TODO: Implement
    # Steps:
    # 1. Calculate running maximum (cummax)
    # 2. Calculate drawdown = (value - cummax) / cummax
    # 3. Return the minimum (most negative) drawdown
    pass


def calculate_sharpe_ratio(
    portfolio_values: pd.Series,
    risk_free_rate: float = 0.02
) -> float:
    """Calculate annualized Sharpe ratio.
    
    Args:
        portfolio_values: Series of daily portfolio values
        risk_free_rate: Annual risk-free rate (default 2%)
        
    Returns:
        Annualized Sharpe ratio
    """
    # TODO: Implement
    # Steps:
    # 1. Calculate daily returns from portfolio values
    # 2. Annualize: mean_return * 252, std * sqrt(252)
    # 3. Sharpe = (annualized_return - risk_free_rate) / annualized_std
    pass

In [ ]:
# =============================================================================
# TASK 5 (continued): VISUALIZATION
# =============================================================================
#
# Create a plot comparing your strategy's cumulative return to the benchmark.
#
# =============================================================================

def plot_cumulative_returns(results: Dict) -> None:
    """Plot cumulative returns of strategy vs benchmark.
    
    Args:
        results: The dict returned by backtester.run()
    """
    # TODO: Implement
    #
    # The results dict contains:
    #   - results['history']: DataFrame with 'date' and 'portfolio_value' columns
    #   - results['benchmark']['daily_values']: DataFrame with 'date' and 'benchmark_value'
    #
    # Plot both on the same chart, with:
    #   - Clear legend
    #   - Proper axis labels
    #   - Title
    #
    pass

In [ ]:
# =============================================================================
# TASK 5 (continued): PRINT RESULTS
# =============================================================================

def print_performance_summary(results: Dict) -> None:
    """Print the required performance summary.
    
    Args:
        results: The dict returned by backtester.run()
    """
    # TODO: Implement to print in this format:
    #
    # ============ PORTFOLIO PERFORMANCE ============
    # Final Portfolio Value:    $X,XXX,XXX.XX
    # Annualized Return:        XX.XX%
    # Maximum Drawdown:         -XX.XX%
    # Sharpe Ratio:             X.XX
    #
    # ============ BENCHMARK (S&P 500) ============
    # Final Value:              $X,XXX,XXX.XX
    # Annualized Return:        XX.XX%
    #
    # ============ RESULT ============
    # Beat Benchmark:           Yes/No
    # Outperformance:           +/-XX.XX%
    # ===============================================
    #
    pass

---

## Part 8: Running the Backtest

Once you've implemented the tasks above, here's how to run the full backtest:

In [ ]:
# =============================================================================
# RUN THE BACKTEST
# =============================================================================
#
# Uncomment and run after implementing all tasks above.
#
# =============================================================================

# # 1. Prepare training data
# X_train, y_train = prepare_training_data(signals)
# print(f"Training data: {len(X_train)} samples")

# # 2. Train model
# model = YourModel()
# model.fit(X_train, y_train)
# print("Model trained!")

# # 3. Initialize backtester
# backtester = Backtester(
#     signals=signals,
#     initial_cash=1_000_000,
#     benchmark_ticker='^SP500TR'
# )

# # 4. Run backtest
# results = backtester.run(
#     model=model,
#     allocate_fn=allocate_cash,
#     start_date='2016-04-01',
#     end_date='2025-12-31',
#     retrain_fn=None,  # Set to retrain_model if you implemented it
#     retrain_frequency=None,  # e.g., 252 for yearly retraining
#     verbose=True
# )

# # 5. Print summary and plot
# print_performance_summary(results)
# plot_cumulative_returns(results)

---

## Part 9: Demo with a Simple Model

Let's run a quick demo to verify the backtester works. We'll create a **small, self-contained dataset** so this demo runs without needing the actual database.

**Demo setup:**
- 100 trading days of data (~5 months, enough for positions to close after 90 days)
- 5 tickers: AAPL, MSFT, GOOGL, AMZN, and ^SP500TR (benchmark)
- 2 dummy feature columns: `momentum_score` and `pe_ratio`
- A "DummyModel" that returns random predictions

This demonstrates the backtester mechanics - **don't expect good results from random predictions!**

In [ ]:
# Reuse the demo-data generator defined in Part 1.
# Create the demo dataset
demo_signals = create_demo_signals()

print("Demo signals table created!")
print(f"Shape: {demo_signals.shape}")
print(f"Symbols: {demo_signals['symbol'].unique().tolist()}")
print(f"Date range: {demo_signals['date'].min().date()} to {demo_signals['date'].max().date()}")
print(f"Columns: {demo_signals.columns.tolist()}")
print()
demo_signals.head(10)

In [ ]:
class DummyModel:
    """A dummy model for testing the backtester infrastructure.
    
    This just returns random predictions - do NOT use for HW4!
    """
    
    def fit(self, X, y):
        # Does nothing
        return self
    
    def predict(self, X):
        # Return random predictions between -0.1 and 0.3
        predictions = np.random.uniform(-0.1, 0.3, size=len(X))
        return pd.Series(predictions, index=X.index)


def simple_allocate(predictions, available_cash, day_data):
    """A simple allocation: invest in top 2 positive predictions."""
    # Only consider positive predictions (exclude benchmark ticker)
    positive = predictions[
        (predictions > 0.05) & 
        (~predictions.index.str.startswith('^'))  # Exclude index tickers
    ]
    if len(positive) == 0:
        return {}
    
    # Take top 2 stocks
    top_picks = positive.nlargest(2)
    
    # Equal weight among selected stocks, keep 20% cash reserve
    dollars_each = available_cash * 0.8 / len(top_picks)
    
    return {ticker: dollars_each for ticker in top_picks.index}

In [ ]:
# Run the demo backtest using our small dummy dataset
print("Running demo backtest with dummy model...")
print("(This uses random predictions - expect variable results!)\n")

# Initialize backtester with DEMO signals (not the real database)
backtester = Backtester(
    signals=demo_signals,  # <-- Using our small dummy dataset
    initial_cash=1_000_000,
    benchmark_ticker='^SP500TR'
)

dummy_model = DummyModel()

# Get the date range from our demo data
demo_start = demo_signals['date'].min().strftime('%Y-%m-%d')
demo_end = demo_signals['date'].max().strftime('%Y-%m-%d')

# Run backtest over the demo period
demo_results = backtester.run(
    model=dummy_model,
    allocate_fn=simple_allocate,
    start_date=demo_start,
    end_date=demo_end,
    feature_columns=['momentum_score', 'pe_ratio'],  # Our dummy features
    verbose=True
)

In [ ]:
# Look at the history DataFrame
demo_results['history'].head(10)

In [ ]:
# Look at some closed positions
if len(demo_results['closed_positions']) > 0:
    print(f"Total closed positions: {len(demo_results['closed_positions'])}")
    demo_results['closed_positions'].head(10)

In [ ]:
# Check benchmark data
print("Benchmark stats:")
for key, value in demo_results['benchmark'].items():
    if key != 'daily_values':
        print(f"  {key}: {value}")

---

## Part 10: Key Data Available to You

Here's a summary of what data is available at each step:

### In `allocate_cash(predictions, available_cash, day_data)`:

| Variable | Type | Description |
|----------|------|-------------|
| `predictions` | `pd.Series` | Predicted 90-day returns, indexed by ticker |
| `available_cash` | `float` | How much cash you can invest |
| `day_data` | `pd.DataFrame` | Today's data for all stocks, indexed by ticker |

### In backtest `results` dict:

| Key | Type | Description |
|-----|------|-------------|
| `'history'` | `pd.DataFrame` | Daily snapshots with `date`, `portfolio_value`, `cash`, etc. |
| `'closed_positions'` | `pd.DataFrame` | All closed trades with P&L |
| `'open_positions'` | `List[Position]` | Positions still open at end |
| `'benchmark'` | `Dict` | Benchmark stats and `daily_values` DataFrame |
| `'final_value'` | `float` | Final portfolio value |
| `'initial_cash'` | `float` | Starting capital |

---

## Summary: What You Need to Do for HW4

1. **Implement `YourModel`** (or use sklearn Pipeline)
   - Must have `.fit(X, y)` and `.predict(X)` methods
   - Choose your algorithm (Ridge, RandomForest, XGBoost, etc.)

2. **Implement `prepare_training_data()`**
   - Calculate 90-day forward return as target
   - Select/engineer features
   - Filter to training period

3. **Implement `allocate_cash()`**
   - Decide which stocks to buy
   - Decide how much to invest in each

4. **Implement performance metrics**
   - `calculate_annualized_return()`
   - `calculate_max_drawdown()`
   - `calculate_sharpe_ratio()`

5. **Implement `plot_cumulative_returns()`**
   - Strategy vs. benchmark comparison

6. **Implement `print_performance_summary()`**
   - Output in the required format

7. **(Optional) Implement `retrain_model()`**
   - For rolling window retraining

**Good luck!**